# DreamerV3 World Model MuJoCo Humanoid (CuPy GPU)
From-scratch DreamerV3 with CuPy (GPU arrays) for full GPU-accelerated forward+backward.
All autograd, neural net layers, RSSM, training logic is our code — CuPy just replaces NumPy as the array backend.

In [ ]:
!nvidia-smi
!pip install cupy-cuda12x gymnasium[mujoco] -q
import os
os.environ['MUJOCO_GL'] = 'egl'
import cupy as cp
print(f'CuPy version: {cp.__version__}')
print(f'GPU: {cp.cuda.runtime.getDeviceProperties(0)["name"]}')
a = cp.random.randn(512, 512, dtype=cp.float32)
b = cp.random.randn(512, 512, dtype=cp.float32)
_ = cp.matmul(a, b)
print('CuPy GPU matmul works!')

In [ ]:
import os
for d in ['nn', 'model', 'agent', 'training']:
    os.makedirs(d, exist_ok=True)
    open(f'{d}/__init__.py', 'w').close()
print('Directories created')

In [ ]:
%%writefile nn/tensor.py
import numpy as np
try:
    import cupy as cp
    _xp = cp
    GPU = True
except ImportError:
    _xp = np
    GPU = False

def to_gpu(arr):
    if GPU and isinstance(arr, np.ndarray): return cp.asarray(arr, dtype=cp.float32)
    return arr

def to_cpu(arr):
    if GPU and isinstance(arr, cp.ndarray): return cp.asnumpy(arr)
    return np.asarray(arr)

def _unbroadcast(grad, shape):
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for i, (g, s) in enumerate(zip(grad.shape, shape)):
        if s == 1 and g != 1:
            grad = grad.sum(axis=i, keepdims=True)
    return grad


class Tensor:
    def __init__(self, data, requires_grad=False, _children=(), _op=''):
        if isinstance(data, (float, int)):
            self.data = _xp.array(data, dtype=_xp.float32)
        elif isinstance(data, np.ndarray):
            self.data = to_gpu(data.astype(np.float32))
        elif GPU and isinstance(data, cp.ndarray):
            self.data = data if data.dtype == cp.float32 else data.astype(cp.float32)
        else:
            self.data = _xp.array(data, dtype=_xp.float32)
        self.requires_grad = requires_grad
        self.grad = None
        self._backward = lambda: None
        self._children = set(_children)
        self._op = _op

    @property
    def shape(self):
        return self.data.shape

    def zero_grad(self):
        self.grad = None

    def detach(self):
        return Tensor(self.data.copy(), requires_grad=False)

    def numpy(self):
        return to_cpu(self.data)

    def backward(self):
        topo = []
        visited = set()
        def build(node):
            if node not in visited:
                visited.add(node)
                for child in node._children:
                    build(child)
                topo.append(node)
        build(self)
        self.grad = _xp.ones_like(self.data)
        for node in reversed(topo):
            node._backward()

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, requires_grad=self.requires_grad or other.requires_grad, _children=(self, other), _op='+')
        def _backward():
            if self.requires_grad:
                g = _unbroadcast(out.grad, self.shape)
                self.grad = g if self.grad is None else self.grad + g
            if other.requires_grad:
                g = _unbroadcast(out.grad, other.shape)
                other.grad = g if other.grad is None else other.grad + g
        out._backward = _backward
        return out

    def __radd__(self, other): return self.__add__(other)

    def __sub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data - other.data, requires_grad=self.requires_grad or other.requires_grad, _children=(self, other), _op='-')
        def _backward():
            if self.requires_grad:
                g = _unbroadcast(out.grad, self.shape)
                self.grad = g if self.grad is None else self.grad + g
            if other.requires_grad:
                g = _unbroadcast(-out.grad, other.shape)
                other.grad = g if other.grad is None else other.grad + g
        out._backward = _backward
        return out

    def __rsub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return other.__sub__(self)

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, requires_grad=self.requires_grad or other.requires_grad, _children=(self, other), _op='*')
        def _backward():
            if self.requires_grad:
                g = _unbroadcast(out.grad * other.data, self.shape)
                self.grad = g if self.grad is None else self.grad + g
            if other.requires_grad:
                g = _unbroadcast(out.grad * self.data, other.shape)
                other.grad = g if other.grad is None else other.grad + g
        out._backward = _backward
        return out

    def __rmul__(self, other): return self.__mul__(other)
    def __neg__(self): return self * (-1)

    def __pow__(self, power):
        out = Tensor(self.data ** power, requires_grad=self.requires_grad, _children=(self,), _op=f'**{power}')
        def _backward():
            if self.requires_grad:
                g = out.grad * power * self.data ** (power - 1)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def matmul(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(_xp.matmul(self.data, other.data), requires_grad=self.requires_grad or other.requires_grad, _children=(self, other), _op='matmul')
        def _backward():
            if self.requires_grad:
                g = _xp.matmul(out.grad, _xp.swapaxes(other.data, -1, -2))
                self.grad = g if self.grad is None else self.grad + g
            if other.requires_grad:
                g = _xp.matmul(_xp.swapaxes(self.data, -1, -2), out.grad)
                other.grad = g if other.grad is None else other.grad + g
        out._backward = _backward
        return out

    def sum(self, axis=None, keepdims=False):
        out = Tensor(self.data.sum(axis=axis, keepdims=keepdims), requires_grad=self.requires_grad, _children=(self,), _op='sum')
        def _backward():
            if self.requires_grad:
                if axis is None:
                    g = _xp.ones_like(self.data) * out.grad
                elif not keepdims:
                    g = _xp.expand_dims(out.grad, axis=axis) * _xp.ones_like(self.data)
                else:
                    g = out.grad * _xp.ones_like(self.data)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def mean(self, axis=None, keepdims=False):
        n = self.data.size if axis is None else self.data.shape[axis]
        return self.sum(axis=axis, keepdims=keepdims) * (1.0 / n)

    def reshape(self, *shape):
        if len(shape) == 1 and isinstance(shape[0], tuple): shape = shape[0]
        out = Tensor(self.data.reshape(shape), requires_grad=self.requires_grad, _children=(self,), _op='reshape')
        def _backward():
            if self.requires_grad:
                self.grad = out.grad.reshape(self.shape) if self.grad is None else self.grad + out.grad.reshape(self.shape)
        out._backward = _backward
        return out

    def sigmoid(self):
        s = 1.0 / (1.0 + _xp.exp(-_xp.clip(self.data, -88, 88)))
        out = Tensor(s, requires_grad=self.requires_grad, _children=(self,), _op='sigmoid')
        def _backward():
            if self.requires_grad:
                g = out.grad * s * (1 - s)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def tanh(self):
        t = _xp.tanh(self.data)
        out = Tensor(t, requires_grad=self.requires_grad, _children=(self,), _op='tanh')
        def _backward():
            if self.requires_grad:
                g = out.grad * (1 - t ** 2)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def relu(self):
        out = Tensor(_xp.maximum(0, self.data), requires_grad=self.requires_grad, _children=(self,), _op='relu')
        def _backward():
            if self.requires_grad:
                g = out.grad * (self.data > 0).astype(_xp.float32)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def silu(self):
        s = 1.0 / (1.0 + _xp.exp(-_xp.clip(self.data, -88, 88)))
        out = Tensor(self.data * s, requires_grad=self.requires_grad, _children=(self,), _op='silu')
        def _backward():
            if self.requires_grad:
                g = out.grad * (s + self.data * s * (1 - s))
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def softmax(self, axis=-1):
        e = _xp.exp(self.data - self.data.max(axis=axis, keepdims=True))
        s = e / e.sum(axis=axis, keepdims=True)
        out = Tensor(s, requires_grad=self.requires_grad, _children=(self,), _op='softmax')
        def _backward():
            if self.requires_grad:
                g = out.grad * s - s * (out.grad * s).sum(axis=axis, keepdims=True)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def log(self):
        out = Tensor(_xp.log(self.data + 1e-8), requires_grad=self.requires_grad, _children=(self,), _op='log')
        def _backward():
            if self.requires_grad:
                g = out.grad / (self.data + 1e-8)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def clamp(self, min_val=None, max_val=None):
        data = self.data.copy()
        if min_val is not None: data = _xp.maximum(data, min_val)
        if max_val is not None: data = _xp.minimum(data, max_val)
        out = Tensor(data, requires_grad=self.requires_grad, _children=(self,), _op='clamp')
        def _backward():
            if self.requires_grad:
                mask = _xp.ones_like(self.data)
                if min_val is not None: mask *= (self.data >= min_val)
                if max_val is not None: mask *= (self.data <= max_val)
                g = out.grad * mask
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def symlog(self):
        out = Tensor(_xp.sign(self.data) * _xp.log(1 + _xp.abs(self.data)), requires_grad=self.requires_grad, _children=(self,), _op='symlog')
        def _backward():
            if self.requires_grad:
                g = out.grad / (1 + _xp.abs(self.data))
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def straight_through(self, hard):
        if isinstance(hard, np.ndarray): hard = to_gpu(hard)
        out = Tensor(hard, requires_grad=self.requires_grad, _children=(self,), _op='st')
        def _backward():
            if self.requires_grad:
                self.grad = out.grad.copy() if self.grad is None else self.grad + out.grad
        out._backward = _backward
        return out

    def layer_norm(self, scale, shift, eps=1e-5):
        mu = self.data.mean(axis=-1, keepdims=True)
        var = self.data.var(axis=-1, keepdims=True)
        x_hat = (self.data - mu) / _xp.sqrt(var + eps)
        y = scale.data * x_hat + shift.data
        out = Tensor(y, requires_grad=self.requires_grad or scale.requires_grad or shift.requires_grad, _children=(self, scale, shift), _op='ln')
        def _backward():
            if out.grad is None: return
            D = self.data.shape[-1]
            inv_std = 1.0 / _xp.sqrt(var + eps)
            dy = out.grad
            if scale.requires_grad:
                sg = (dy * x_hat).sum(axis=tuple(range(dy.ndim - 1)))
                scale.grad = sg if scale.grad is None else scale.grad + sg
            if shift.requires_grad:
                bg = dy.sum(axis=tuple(range(dy.ndim - 1)))
                shift.grad = bg if shift.grad is None else shift.grad + bg
            if self.requires_grad:
                dx_hat = dy * scale.data
                g = inv_std * (dx_hat - dx_hat.mean(axis=-1, keepdims=True) - x_hat * (dx_hat * x_hat).mean(axis=-1, keepdims=True))
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out


def cat(tensors, axis=0):
    data = _xp.concatenate([t.data for t in tensors], axis=axis)
    out = Tensor(data, requires_grad=any(t.requires_grad for t in tensors), _children=tuple(tensors), _op='cat')
    def _backward():
        if out.grad is None: return
        sections = _xp.cumsum(_xp.array([t.shape[axis] for t in tensors[:-1]]))
        grads = _xp.split(out.grad, to_cpu(sections).astype(int).tolist(), axis=axis)
        for t, g in zip(tensors, grads):
            if t.requires_grad:
                t.grad = g.copy() if t.grad is None else t.grad + g
    out._backward = _backward
    return out


def one_hot(indices, num_classes):
    flat = indices.reshape(-1)
    oh = np.zeros((flat.size, num_classes), dtype=np.float32)
    oh[np.arange(flat.size), flat] = 1.0
    return oh.reshape(indices.shape + (num_classes,))

In [ ]:
%%writefile nn/linear.py
import numpy as np
from nn.tensor import Tensor, _xp

class Linear:
    def __init__(self, in_features, out_features, bias=True):
        scale = (2.0 / in_features) ** 0.5
        self.weight = Tensor(_xp.random.randn(in_features, out_features).astype(_xp.float32) * scale, requires_grad=True)
        self.bias = Tensor(_xp.zeros(out_features, dtype=_xp.float32), requires_grad=True) if bias else None

    def __call__(self, x):
        out = x.matmul(self.weight)
        if self.bias is not None: out = out + self.bias
        return out

    def parameters(self):
        return [self.weight] + ([self.bias] if self.bias else [])

class LayerNorm:
    def __init__(self, dim):
        self.scale = Tensor(_xp.ones(dim, dtype=_xp.float32), requires_grad=True)
        self.shift = Tensor(_xp.zeros(dim, dtype=_xp.float32), requires_grad=True)
    def __call__(self, x):
        return x.layer_norm(self.scale, self.shift)
    def parameters(self):
        return [self.scale, self.shift]

In [ ]:
%%writefile nn/mlp.py
from nn.linear import Linear, LayerNorm

class MLP:
    def __init__(self, sizes, activation='silu'):
        self.activation = activation
        self.layers = [Linear(sizes[i], sizes[i+1]) for i in range(len(sizes)-1)]
        self.norms = [LayerNorm(sizes[i+1]) for i in range(len(sizes)-2)]
    def __call__(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = self.norms[i](x)
                x = x.silu() if self.activation == 'silu' else x.relu()
        return x
    def parameters(self):
        p = []
        for l in self.layers: p.extend(l.parameters())
        for n in self.norms: p.extend(n.parameters())
        return p

In [ ]:
%%writefile nn/optimizer.py
from nn.tensor import _xp

class AdamW:
    def __init__(self, parameters, lr=3e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01, grad_clip=100.0):
        self.parameters = parameters
        self.lr, self.eps, self.weight_decay, self.grad_clip = lr, eps, weight_decay, grad_clip
        self.beta1, self.beta2 = betas
        self.t = 0
        self.m = [_xp.zeros_like(p.data) for p in parameters]
        self.v = [_xp.zeros_like(p.data) for p in parameters]

    def step(self):
        self.t += 1
        if self.grad_clip > 0:
            norm = float(_xp.sqrt(sum(_xp.sum(p.grad**2) for p in self.parameters if p.grad is not None)))
            if norm > self.grad_clip:
                for p in self.parameters:
                    if p.grad is not None: p.grad = p.grad * (self.grad_clip / (norm + 1e-6))
        for i, p in enumerate(self.parameters):
            if p.grad is None: continue
            self.m[i] = self.beta1 * self.m[i] + (1-self.beta1) * p.grad
            self.v[i] = self.beta2 * self.v[i] + (1-self.beta2) * (p.grad**2)
            mh = self.m[i] / (1-self.beta1**self.t)
            vh = self.v[i] / (1-self.beta2**self.t)
            p.data -= self.lr * self.weight_decay * p.data
            p.data -= self.lr * mh / (_xp.sqrt(vh) + self.eps)

    def zero_grad(self):
        for p in self.parameters: p.zero_grad()

In [ ]:
%%writefile nn/gru_cell.py
from nn.tensor import Tensor, cat, _xp
from nn.linear import Linear

class GRUCell:
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        self.reset_gate = Linear(input_size + hidden_size, hidden_size)
        self.update_gate = Linear(input_size + hidden_size, hidden_size)
        self.candidate = Linear(input_size + hidden_size, hidden_size)
    def __call__(self, x, h):
        if h is None: h = Tensor(_xp.zeros((x.shape[0], self.hidden_size), dtype=_xp.float32))
        combined = cat([x, h], axis=1)
        r = self.reset_gate(combined).sigmoid()
        z = self.update_gate(combined).sigmoid()
        candidate = self.candidate(cat([x, r * h], axis=1)).tanh()
        return z * h + (1 - z) * candidate
    def parameters(self):
        p = []
        for l in [self.reset_gate, self.update_gate, self.candidate]: p.extend(l.parameters())
        return p

In [ ]:
%%writefile model/encoder.py
from nn.mlp import MLP
class Encoder:
    def __init__(self, obs_size, hidden_size, latent_size):
        self.net = MLP([obs_size, hidden_size, hidden_size, latent_size])
    def __call__(self, obs): return self.net(obs)
    def parameters(self): return self.net.parameters()

In [ ]:
%%writefile model/decoder.py
from nn.mlp import MLP
from nn.tensor import cat
class Decoder:
    def __init__(self, state_size, obs_size, hidden_size):
        self.net = MLP([state_size, hidden_size, hidden_size, obs_size])
    def __call__(self, h, z): return self.net(cat([h, z], axis=1))
    def parameters(self): return self.net.parameters()

In [ ]:
%%writefile model/reward_model.py
from nn.mlp import MLP
from nn.tensor import cat
class RewardModel:
    def __init__(self, state_size, hidden_size):
        self.net = MLP([state_size, hidden_size, hidden_size, 1])
    def __call__(self, h, z): return self.net(cat([h, z], axis=1))
    def parameters(self): return self.net.parameters()

In [ ]:
%%writefile model/continue_model.py
from nn.mlp import MLP
from nn.tensor import cat
class ContinueModel:
    def __init__(self, state_size, hidden_size):
        self.net = MLP([state_size, hidden_size, hidden_size, 1])
    def __call__(self, h, z): return self.net(cat([h, z], axis=1)).sigmoid()
    def parameters(self): return self.net.parameters()

In [ ]:
%%writefile model/rssm.py
import numpy as np
from nn.tensor import Tensor, cat, one_hot, _xp, to_cpu
from nn.linear import Linear
from nn.mlp import MLP
from nn.gru_cell import GRUCell

class RSSM:
    def __init__(self, obs_size, action_size, hidden_size, stoch_size, stoch_classes):
        self.hidden_size = hidden_size
        self.stoch_size = stoch_size
        self.stoch_classes = stoch_classes
        stoch_flat = stoch_size * stoch_classes
        self.pre_gru = Linear(stoch_flat + action_size, hidden_size)
        self.gru = GRUCell(hidden_size, hidden_size)
        self.prior = MLP([hidden_size, hidden_size, stoch_flat])
        self.posterior = MLP([hidden_size + obs_size, hidden_size, stoch_flat])

    def forward(self, prev_h, prev_z, action, encoded_obs=None):
        if prev_z is None: prev_z = Tensor(_xp.zeros((action.shape[0], self.stoch_size * self.stoch_classes), dtype=_xp.float32))
        if prev_h is None: prev_h = Tensor(_xp.zeros((action.shape[0], self.hidden_size), dtype=_xp.float32))
        gru_input = self.pre_gru(cat([prev_z, action], axis=1)).silu()
        h = self.gru(gru_input, prev_h)
        prior_logits = self.prior(h).reshape(-1, self.stoch_size, self.stoch_classes)
        if encoded_obs is not None:
            post_logits = self.posterior(cat([h, encoded_obs], axis=1)).reshape(-1, self.stoch_size, self.stoch_classes)
            logits = post_logits
        else:
            post_logits = None
            logits = prior_logits
        probs = logits.softmax(axis=-1)
        # Uniform mix: 99% network + 1% uniform (prevents near-deterministic categoricals)
        uniform = _xp.ones_like(probs.data) / self.stoch_classes
        mixed = 0.99 * probs.data + 0.01 * uniform
        probs = probs.straight_through(mixed)
        probs_np = to_cpu(probs.data)
        indices = np.array([[np.random.choice(self.stoch_classes, p=probs_np[b,d]) for d in range(self.stoch_size)] for b in range(probs_np.shape[0])])
        z = probs.straight_through(one_hot(indices, self.stoch_classes))
        return h, z.reshape(-1, self.stoch_size * self.stoch_classes), prior_logits, post_logits

    def parameters(self):
        p = self.pre_gru.parameters()
        p.extend(self.gru.parameters())
        p.extend(self.prior.parameters())
        p.extend(self.posterior.parameters())
        return p

In [ ]:
%%writefile model/world_model.py
from nn.tensor import Tensor, cat
from model.encoder import Encoder
from model.rssm import RSSM
from model.decoder import Decoder
from model.reward_model import RewardModel
from model.continue_model import ContinueModel

class WorldModel:
    def __init__(self, obs_size, action_size, hidden_size, stoch_size, stoch_classes):
        state_size = hidden_size + stoch_size * stoch_classes
        self.encoder = Encoder(obs_size, hidden_size, hidden_size)
        self.rssm = RSSM(hidden_size, action_size, hidden_size, stoch_size, stoch_classes)
        self.decoder = Decoder(state_size, obs_size, hidden_size)
        self.reward_model = RewardModel(state_size, hidden_size)
        self.continue_model = ContinueModel(state_size, hidden_size)

    def forward(self, observations, actions, rewards, continues):
        h, z = None, None
        all_h, all_z, all_prior, all_post = [], [], [], []
        for t in range(len(observations)):
            encoded = self.encoder(observations[t].symlog())
            h, z, prior, post = self.rssm.forward(h, z, actions[t], encoded_obs=encoded)
            all_h.append(h); all_z.append(z); all_prior.append(prior); all_post.append(post)
        H, Z = cat(all_h, axis=0), cat(all_z, axis=0)
        recon_loss = ((self.decoder(H, Z) - cat(observations, axis=0).symlog()) ** 2).mean()
        reward_loss = ((self.reward_model(H, Z) - cat(rewards, axis=0).symlog()) ** 2).mean()
        continue_loss = ((self.continue_model(H, Z) - cat(continues, axis=0)) ** 2).mean()
        prior_p = cat(all_prior, axis=0).softmax(axis=-1)
        post_p = cat(all_post, axis=0).softmax(axis=-1)
        # KL balancing: 0.8 trains prior toward posterior, 0.2 trains posterior toward prior
        kl_to_prior = (post_p.detach() * (post_p.detach().log() - prior_p.log())).sum(axis=-1).mean()
        kl_to_post = (post_p * (post_p.log() - prior_p.detach().log())).sum(axis=-1).mean()
        kl_loss = 0.5 * kl_to_prior.clamp(min_val=1.0) + 0.1 * kl_to_post.clamp(min_val=1.0)
        return 1.0 * recon_loss + 1.0 * reward_loss + 1.0 * continue_loss + kl_loss

    def parameters(self):
        p = self.encoder.parameters()
        p.extend(self.rssm.parameters())
        p.extend(self.decoder.parameters())
        p.extend(self.reward_model.parameters())
        p.extend(self.continue_model.parameters())
        return p

In [ ]:
%%writefile agent/actor.py
import numpy as np
from nn.mlp import MLP
from nn.tensor import Tensor, cat, _xp

class Actor:
    def __init__(self, state_size, action_size, hidden_size):
        self.action_size = action_size
        self.net = MLP([state_size, hidden_size, hidden_size, action_size * 2])
        self.net.layers[-1].weight.data *= 0.01
        if self.net.layers[-1].bias is not None:
            self.net.layers[-1].bias.data *= 0.0

    def __call__(self, h, z, deterministic=False):
        out = self.net(cat([h, z], axis=1))
        mean = Tensor(out.data[..., :self.action_size], requires_grad=out.requires_grad, _children=(out,), _op='split_mean')
        log_std_raw = out.data[..., self.action_size:]
        log_std = _xp.clip(log_std_raw, -5.0, 2.0)
        std = _xp.exp(log_std)
        def _backward_split():
            if out.requires_grad:
                full_grad = _xp.zeros_like(out.data)
                if mean.grad is not None:
                    full_grad[..., :self.action_size] = mean.grad
                out.grad = full_grad if out.grad is None else out.grad + full_grad
        mean._backward = _backward_split
        if deterministic:
            return mean.tanh(), Tensor(0.0)
        noise = _xp.array(np.random.randn(*std.shape).astype(np.float32))
        if hasattr(_xp, 'asarray'): noise = _xp.asarray(noise)
        pre_tanh = Tensor(mean.data + std * noise, requires_grad=mean.requires_grad, _children=(mean,), _op='reparam')
        def _backward_reparam():
            if mean.requires_grad:
                mean.grad = pre_tanh.grad.copy() if mean.grad is None else mean.grad + pre_tanh.grad
        pre_tanh._backward = _backward_reparam
        action = pre_tanh.tanh()
        entropy = Tensor(_xp.sum(log_std + 0.5 * np.log(2 * np.pi * np.e), axis=-1, keepdims=True).mean())
        return action, entropy

    def parameters(self): return self.net.parameters()

In [ ]:
%%writefile agent/critic.py
import copy
from nn.mlp import MLP
from nn.linear import Linear, LayerNorm
from nn.tensor import Tensor, cat, _xp

class Critic:
    def __init__(self, state_size, hidden_size):
        self.net = MLP([state_size, hidden_size, hidden_size, 1])
        self.net.layers[-1].weight.data *= 0.01
        if self.net.layers[-1].bias is not None:
            self.net.layers[-1].bias.data *= 0.0
        self.target_params = [_xp.copy(p.data) for p in self.parameters()]

    def __call__(self, h, z): return self.net(cat([h, z], axis=1))

    def target_value(self, h, z):
        params = self.parameters()
        saved = [_xp.copy(p.data) for p in params]
        for p, tp in zip(params, self.target_params): p.data = tp
        with_target = self.net(cat([h, z], axis=1))
        val = Tensor(with_target.data.copy())
        for p, s in zip(params, saved): p.data = s
        return val

    def update_target(self, tau=0.02):
        for i, p in enumerate(self.parameters()):
            self.target_params[i] = (1 - tau) * self.target_params[i] + tau * p.data

    def parameters(self): return self.net.parameters()

In [ ]:
%%writefile training/replay_buffer.py
import numpy as np

class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.capacity = capacity
        self.observations, self.actions, self.rewards, self.continues = [], [], [], []
    def add(self, obs, action, reward, done):
        self.observations.append(obs); self.actions.append(action)
        self.rewards.append(reward); self.continues.append(0.0 if done else 1.0)
        if len(self.observations) > self.capacity:
            self.observations.pop(0); self.actions.pop(0); self.rewards.pop(0); self.continues.pop(0)
    def sample(self, batch_size, seq_len):
        max_start = len(self.observations) - seq_len
        if max_start < 1: return None
        idx = np.random.randint(0, max_start, size=batch_size)
        obs_s, act_s, rew_s, con_s = [], [], [], []
        for t in range(seq_len):
            obs_s.append(np.stack([self.observations[i+t] for i in idx]).astype(np.float32))
            act_s.append(np.stack([self.actions[i+t] for i in idx]).astype(np.float32))
            rew_s.append(np.stack([[self.rewards[i+t]] for i in idx]).astype(np.float32))
            con_s.append(np.stack([[self.continues[i+t]] for i in idx]).astype(np.float32))
        return obs_s, act_s, rew_s, con_s
    def __len__(self): return len(self.observations)

In [ ]:
# === SPEED TEST: CuPy vs NumPy ===
import sys, time
sys.path.insert(0, '/content')
import numpy as np
from nn.tensor import Tensor, GPU
from nn.mlp import MLP

print(f'GPU backend active: {GPU}')

net = MLP([376, 512, 512, 512])
x = Tensor(np.random.randn(32, 376).astype(np.float32), requires_grad=True)

# Warmup
out = net(x); out.sum().backward()

# Benchmark forward+backward
t0 = time.time()
for _ in range(50):
    for p in net.parameters(): p.zero_grad()
    x = Tensor(np.random.randn(32, 376).astype(np.float32), requires_grad=True)
    out = net(x)
    out.sum().backward()
elapsed = time.time() - t0
print(f'50 forward+backward passes: {elapsed:.2f}s ({elapsed/50*1000:.1f}ms each)')
print(f'Estimated epoch time (100 WM steps): {elapsed/50*100:.0f}s')

In [ ]:
import gymnasium as gym
from nn.tensor import Tensor, cat, to_cpu, _xp
from nn.optimizer import AdamW
from model.world_model import WorldModel
from agent.actor import Actor
from agent.critic import Critic
from training.replay_buffer import ReplayBuffer

# === CUSTOM HUMANOID WITH BETTER VISUALS ===
import gymnasium.envs.mujoco
import inspect, pathlib
src = pathlib.Path(inspect.getfile(gymnasium.envs.mujoco)).parent / 'assets' / 'humanoid.xml'
xml = open(src).read()
# Swap the brown/orange for a dark metallic blue body
xml = xml.replace('rgba=".7 .3 .1', 'rgba=".15 .25 .45')
xml = xml.replace('rgba=".8 .6 .4', 'rgba=".3 .4 .55')
xml = xml.replace('rgba=".5 .1 .1', 'rgba=".1 .15 .35')
# Brighter floor
xml = xml.replace('rgb1=".8 .9 .8"', 'rgb1=".85 .9 .95"')
xml = xml.replace('rgb2=".4 .6 .4"', 'rgb2=".55 .65 .75"')
with open(src, 'w') as f: f.write(xml)
print('Custom humanoid model saved (overwrote default XML)')

ENV_NAME = 'Humanoid-v4'
env = gym.make(ENV_NAME)
obs_size = env.observation_space.shape[0]
action_size = env.action_space.shape[0]
print(f'{ENV_NAME}: obs_size={obs_size}, action_size={action_size}')

hidden_size = 512
stoch_size = 32
stoch_classes = 32
state_size = hidden_size + stoch_size * stoch_classes

world_model = WorldModel(obs_size, action_size, hidden_size, stoch_size, stoch_classes)
actor = Actor(state_size, action_size, hidden_size=512)
critic = Critic(state_size, hidden_size=512)

wm_opt = AdamW(world_model.parameters(), lr=1e-4, grad_clip=1000.0)
actor_opt = AdamW(actor.parameters(), lr=8e-5, grad_clip=1000.0)
critic_opt = AdamW(critic.parameters(), lr=8e-5, grad_clip=1000.0)
buffer = ReplayBuffer(capacity=100000)

print(f'Params: WM={sum(p.data.size for p in world_model.parameters()):,} Actor={sum(p.data.size for p in actor.parameters()):,}')
print(f'State size: {state_size}')
print('Ready!')

In [ ]:
def collect_data(env, buffer, wm=None, actor=None, num_steps=1000, explore_noise=0.3):
    obs, _ = env.reset()
    h, z = None, None
    for _ in range(num_steps):
        if actor and wm and np.random.rand() > 0.3:
            obs_t = Tensor(obs.reshape(1,-1).astype(np.float32))
            encoded = wm.encoder(obs_t.symlog())
            prev_a = Tensor(np.zeros((1, action_size), dtype=np.float32))
            h, z, _, _ = wm.rssm.forward(h, z, prev_a, encoded_obs=encoded)
            action, _ = actor(h.detach(), z.detach())
            act_np = to_cpu(action.data)[0]
            act_np += np.random.randn(action_size).astype(np.float32) * explore_noise
            act_np = np.clip(act_np, -1.0, 1.0)
        else:
            act_np = env.action_space.sample()
        next_obs, reward, term, trunc, info = env.step(act_np)
        shaped_reward = reward + 5.0 * info.get('x_velocity', 0)
        buffer.add(obs, act_np, shaped_reward, term or trunc)
        obs = next_obs
        if term or trunc:
            obs, _ = env.reset()
            h, z = None, None

def train_world_model(wm, opt, buffer, batch_size=16, seq_len=32):
    batch = buffer.sample(batch_size, seq_len)
    if batch is None: return None
    obs_s, act_s, rew_s, con_s = batch
    opt.zero_grad()
    loss = wm.forward([Tensor(o) for o in obs_s], [Tensor(a) for a in act_s],
                      [Tensor(r) for r in rew_s], [Tensor(c) for c in con_s])
    loss.backward()
    opt.step()
    return float(to_cpu(loss.data))

def train_actor_critic(wm, actor, critic, actor_opt, critic_opt, buffer, horizon=15, batch_size=16):
    GAMMA = 0.997
    LAMBDA = 0.95
    ENTROPY_SCALE = 3e-4
    batch = buffer.sample(batch_size, 1)
    if batch is None: return
    encoded = wm.encoder(Tensor(batch[0][0]).symlog())
    h, z, _, _ = wm.rssm.forward(None, None, Tensor(batch[1][0]), encoded_obs=encoded)
    h, z = h.detach(), z.detach()
    imagined_h, imagined_z = [h], [z]
    rewards, continues, actions, entropies = [], [], [], []
    for _ in range(horizon):
        action, entropy = actor(h, z)
        actions.append(action)
        entropies.append(entropy)
        h, z, _, _ = wm.rssm.forward(h, z, action, encoded_obs=None)
        h, z = h.detach(), z.detach()
        rewards.append(wm.reward_model(h, z))
        continues.append(wm.continue_model(h, z))
        imagined_h.append(h); imagined_z.append(z)
    # Compute target values using EMA critic (no gradient)
    target_vals = [critic.target_value(imagined_h[t], imagined_z[t]) for t in range(horizon + 1)]
    # TD-lambda returns (backwards)
    returns = [None] * horizon
    last_return = rewards[-1].detach() + continues[-1].detach() * GAMMA * target_vals[horizon]
    returns[-1] = last_return
    for t in reversed(range(horizon - 1)):
        v_next = target_vals[t + 1]
        r_t = rewards[t].detach()
        c_t = continues[t].detach()
        returns[t] = r_t + c_t * GAMMA * (LAMBDA * returns[t + 1] + (1 - LAMBDA) * v_next)
    # Critic loss (online critic vs lambda returns)
    values = [critic(imagined_h[t], imagined_z[t]) for t in range(horizon)]
    critic_loss = Tensor(0.0)
    for t in range(horizon):
        critic_loss = critic_loss + ((values[t] - returns[t]) ** 2).mean()
    critic_opt.zero_grad(); critic_loss.backward(); critic_opt.step()
    critic.update_target(tau=0.02)
    # Percentile return normalization (5th-95th)
    all_returns = _xp.concatenate([r.data.flatten() for r in returns])
    pct5 = float(_xp.percentile(all_returns, 5))
    pct95 = float(_xp.percentile(all_returns, 95))
    scale = max(1.0, pct95 - pct5)
    # Actor loss: maximize normalized returns + entropy bonus
    actor_loss = Tensor(0.0)
    for t in range(horizon):
        normalized_return = (returns[t].detach() * (-1.0 / scale))
        actor_loss = actor_loss + normalized_return.mean()
        actor_loss = actor_loss + (-ENTROPY_SCALE) * entropies[t]
    actor_opt.zero_grad(); actor_loss.backward(); actor_opt.step()

def evaluate(env, wm, actor, num_episodes=3):
    total = []
    for _ in range(num_episodes):
        obs, _ = env.reset()
        h, z = None, None
        ep_reward = 0
        for _ in range(1000):
            obs_t = Tensor(obs.reshape(1,-1).astype(np.float32))
            encoded = wm.encoder(obs_t.symlog())
            prev_a = Tensor(np.zeros((1, action_size), dtype=np.float32))
            h, z, _, _ = wm.rssm.forward(h, z, prev_a, encoded_obs=encoded)
            action, _ = actor(h.detach(), z.detach(), deterministic=True)
            act_np = np.clip(to_cpu(action.data)[0], -1, 1)
            obs, reward, term, trunc, _ = env.step(act_np)
            ep_reward += reward
            if term or trunc: break
        total.append(ep_reward)
    return sum(total)/len(total)

print('Functions defined!')

In [ ]:
# === QUICK TEST: 20 epochs (~20 min) to verify walking signal ===
# If reward trends UP past 250-300 by epoch 15+, you're on track. Then run full training.
# If it plateaus at 150-200, something is still wrong.

print(f'Collecting initial random data...')
collect_data(env, buffer, num_steps=5000)
print(f'Buffer: {len(buffer)}')

baseline = evaluate(env, world_model, actor, num_episodes=3)
print(f'Before training (random): {baseline:.1f}')
print()

WM_WARMUP = 10
TOTAL_EPOCHS = 25  # Quick test
ACTOR_RATIO = 3    # More actor updates

print(f'--- QUICK TEST: {TOTAL_EPOCHS} epochs, WM warmup={WM_WARMUP} ---')
print()

best_reward = baseline
start_time = time.time()
for epoch in range(TOTAL_EPOCHS):
    epoch_start = time.time()
    wm_losses = []
    train_actor = (epoch >= WM_WARMUP)

    for step in range(100):
        wm_loss = train_world_model(world_model, wm_opt, buffer)
        if wm_loss is not None: wm_losses.append(wm_loss)
        if train_actor and step % ACTOR_RATIO == 0:
            train_actor_critic(world_model, actor, critic, actor_opt, critic_opt, buffer)

    avg_wm = sum(wm_losses)/len(wm_losses) if wm_losses else 0
    phase = 'WM+Actor' if train_actor else 'WM only'
    avg_reward = evaluate(env, world_model, actor, num_episodes=3)
    if avg_reward > best_reward: best_reward = avg_reward

    elapsed = time.time() - epoch_start
    total_t = time.time() - start_time
    print(f'epoch {epoch:2d} [{phase:9s}]: wm_loss={avg_wm:.4f}, reward={avg_reward:.1f}, '
          f'best={best_reward:.1f}, buffer={len(buffer)}, time={elapsed:.0f}s (total {total_t:.0f}s)')

    collect_data(env, buffer, world_model if train_actor else None,
                 actor if train_actor else None, num_steps=500, explore_noise=0.4)

print(f'\n=== QUICK TEST DONE ===')
print(f'Best reward: {best_reward:.1f}')
print(f'If best > 250, run FULL TRAINING cell below. If stuck ~150-200, needs more tuning.')

In [ ]:
# === FULL TRAINING: Run ONLY if quick test shows reward trending up ===
# Continues from quick test (models not reset)

FULL_EPOCHS = 80
ACTOR_RATIO = 5

print(f'--- FULL TRAINING: {FULL_EPOCHS} more epochs ---')
print()

start_time = time.time()
for epoch in range(FULL_EPOCHS):
    epoch_start = time.time()
    wm_losses = []

    for step in range(100):
        wm_loss = train_world_model(world_model, wm_opt, buffer)
        if wm_loss is not None: wm_losses.append(wm_loss)
        if step % ACTOR_RATIO == 0:
            train_actor_critic(world_model, actor, critic, actor_opt, critic_opt, buffer)

    avg_wm = sum(wm_losses)/len(wm_losses) if wm_losses else 0
    avg_reward = evaluate(env, world_model, actor, num_episodes=3)
    if avg_reward > best_reward: best_reward = avg_reward

    elapsed = time.time() - epoch_start
    total_t = time.time() - start_time
    print(f'epoch {epoch:2d}: wm_loss={avg_wm:.4f}, reward={avg_reward:.1f}, '
          f'best={best_reward:.1f}, buffer={len(buffer)}, time={elapsed:.0f}s (total {total_t:.0f}s)')

    collect_data(env, buffer, world_model, actor, num_steps=500, explore_noise=0.3)

final = evaluate(env, world_model, actor, num_episodes=5)
print(f'\nFinal avg reward = {final:.1f} (best: {best_reward:.1f})')
print(f'Total time: {time.time()-start_time:.1f}s')

In [ ]:
# === RENDER VIDEO ===
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

render_env = gym.make(ENV_NAME, render_mode='rgb_array')
obs, _ = render_env.reset()
h, z = None, None
frames = []
ep_reward = 0

for step in range(500):
    frames.append(render_env.render())
    obs_t = Tensor(obs.reshape(1,-1).astype(np.float32))
    encoded = world_model.encoder(obs_t.symlog())
    prev_a = Tensor(np.zeros((1, action_size), dtype=np.float32))
    h, z, _, _ = world_model.rssm.forward(h, z, prev_a, encoded_obs=encoded)
    action, _ = actor(h.detach(), z.detach(), deterministic=True)
    act_np = np.clip(to_cpu(action.data)[0], -1, 1)
    obs, reward, term, trunc, _ = render_env.step(act_np)
    ep_reward += reward
    if term or trunc: break

render_env.close()
print(f'Episode: {len(frames)} frames, reward={ep_reward:.1f}')

fig, ax = plt.subplots(figsize=(6,6))
ax.axis('off')
img = ax.imshow(frames[0])
def animate(i): img.set_data(frames[i]); return [img]
anim = animation.FuncAnimation(fig, animate, frames=len(frames), interval=33, blit=True)
plt.close()
HTML(anim.to_jshtml())